In [5]:
# retrain_scaler.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# --- Load ZIP Code Mapping ---
def load_zip_mapping(filepath='../data/zip_data_prefix_mapping.csv'):
    zip_mapping = pd.read_csv(filepath)
    return dict(zip(zip_mapping['zip_prefix'], zip_mapping['addr_state_actual']))

# --- Data Preprocessing ---
def load_and_preprocess_data(filepath):
    data = pd.read_csv(filepath)
    data.fillna(data.median(numeric_only=True), inplace=True)
    data.fillna(0, inplace=True)
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in data.select_dtypes(include=[np.number]).columns:
        data[col] = data[col].clip(lower=-1e10, upper=1e10)
    data.fillna(data.median(numeric_only=True), inplace=True)
    
    # One-hot encode categorical columns
    categorical_cols = ['addr_state', 'grade', 'home_ownership', 'verification_status', 'job_category']
    for col in categorical_cols:
        if col in data.columns:
            dummies = pd.get_dummies(data[col], prefix=col, drop_first=False)
            data = pd.concat([data, dummies], axis=1)
            data = data.drop(columns=[col])
    
    return data

# --- Fraud Detection Functions ---
def detect_geo_fraud(data, zip_mapping):
    if 'geo_fraud' not in data.columns:
        data['geo_fraud'] = 0
    state_cols = [col for col in data.columns if 'addr_state_' in col]
    if state_cols and 'zip_code' in data.columns:
        for index, row in data.iterrows():
            zip_prefix = int(str(row['zip_code'])[:3]) if str(row['zip_code'])[:3].isdigit() else -1
            expected_state = zip_mapping.get(zip_prefix, 'Unknown')
            if expected_state != 'Unknown':
                active_state = next((col.split('_')[-1] for col in state_cols if row[col] == 1), None)
                if active_state and active_state != expected_state:
                    data.at[index, 'geo_fraud'] = 1
    return data

def detect_loan_stacking(data):
    required_cols = ['loan_amnt', 'annual_inc', 'dti']
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for loan stacking detection: {missing_cols}")
    duplicate_mask = data.duplicated(subset=['loan_amnt', 'annual_inc', 'dti'], keep=False)
    data['loan_stacking_fraud'] = duplicate_mask.astype(int)
    return data

def detect_dti_fraud(data):
    required_cols = ['dti', 'loan_amnt', 'annual_inc']
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for DTI fraud detection: {missing_cols}")
    data['adjusted_dti'] = (data['dti'] + (data['loan_amnt'] / data['annual_inc'].replace(0, 1e-10))) * 100
    data['adjusted_dti'] = data['adjusted_dti'].clip(upper=1e10)
    data['dti_fraud'] = data['adjusted_dti'].apply(lambda x: 1 if x > 50 else 0)
    data['dti_high_risk'] = data['adjusted_dti'].apply(lambda x: 1 if 43 <= x <= 50 else 0)
    return data

# --- Feature Engineering ---
def add_features(data, zip_mapping):
    data['geo_mismatch'] = 0
    state_cols = [col for col in data.columns if 'addr_state_' in col]
    if state_cols and 'zip_code' in data.columns:
        for index, row in data.iterrows():
            zip_prefix = int(str(row['zip_code'])[:3]) if str(row['zip_code'])[:3].isdigit() else -1
            expected_state = zip_mapping.get(zip_prefix, 'Unknown')
            if expected_state != 'Unknown':
                active_state = next((col.split('_')[-1] for col in state_cols if row[col] == 1), None)
                if active_state and active_state != expected_state:
                    data.at[index, 'geo_mismatch'] = 1
    
    required_cols = ['loan_amnt', 'annual_inc', 'dti']
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for feature engineering: {missing_cols}")
    data['loan_stacking_count'] = data.groupby(['loan_amnt', 'annual_inc', 'dti'])['loan_amnt'].transform('count')
    
    data['loan_to_income'] = data['loan_amnt'] / data['annual_inc'].replace(0, 1e-10)
    data['dti_revol_util'] = data['dti'] * data.get('revol_util', 0)
    
    return data

# --- Main Function to Retrain Scaler ---
def retrain_scaler():
    # Step 1: Load and preprocess data
    zip_mapping = load_zip_mapping()
    data = load_and_preprocess_data('../data/final_data_cleaned.csv')
    print(f"Columns in final_data_cleaned.csv after preprocessing: {list(data.columns)}")
    
    # Step 2: Apply fraud detection functions
    data = detect_loan_stacking(data)
    data = detect_geo_fraud(data, zip_mapping)
    data = detect_dti_fraud(data)
    
    # Step 3: Add features
    data = add_features(data, zip_mapping)
    
    # Step 4: Prepare data (mimic prepare_data_for_training)
    data['fraud_prediction'] = ((data['geo_fraud'] == 1) | 
                               (data['loan_stacking_fraud'] == 1) | 
                               (data['dti_fraud'] == 1)).astype(int)
    
    exclude_cols = ['fraud_prediction', 'geo_fraud', 'loan_stacking_fraud', 'dti_fraud', 'iso_fraud', 'dti_high_risk', 'adjusted_dti']
    # Include numeric and boolean columns (for one-hot encoded features)
    X = data.select_dtypes(include=[np.number, bool]).drop(columns=exclude_cols, errors='ignore')
    
    print(f"Total features before selection: {len(X.columns)}")
    print(f"Feature names: {list(X.columns)}")
    
    # Step 5: Feature selection
    if len(X.columns) < 96:
        raise ValueError(f"Only {len(X.columns)} features available, need 96")
    selected_features = X.columns[:96].tolist()  # Select the first 96 features
    X_selected = X[selected_features]
    
    # Step 6: Retrain the scaler
    scaler = StandardScaler()
    scaler.fit(X_selected)
    
    # Step 7: Save the scaler
    joblib.dump(scaler, '../models/scaler_with_fraud_data_refactored.pkl')
    print("Scaler retrained and saved to ../models/scaler_with_fraud_data_refactored.pkl")
    print(f"Selected features: {list(selected_features)}")

if __name__ == "__main__":
    retrain_scaler()

Columns in final_data_cleaned.csv after preprocessing: ['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'emp_length', 'annual_inc', 'zip_code', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d', 'mort_acc', 'credit_age_years', 'issue_year', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G', 'sub_grade_A2', 'sub_grade_A3', 'sub_grade_A4', 'sub_grade_A5', 'sub_grade_B1', 'sub_grade_B2', 'sub_grade_B3', 'sub_grade_B4', 'sub_grade_B5', 'sub_grade_C1', 'sub_grade_C2', 'sub_grade_C3', 'sub_grade_C4', 'sub_grade_C5', 'sub_grade_D1', 'sub_grade_D2', 'sub_grade_D3', 'sub_grade_D4', 'sub_grade_D5', 'sub_grade_E1', 'sub_grade_E2', 'sub_grade_E3', 'sub_grade_E4', 'sub_grade_E5', 'sub_grade_F1', 'sub_grade_F2', 'sub_grade_F3', 'sub_grade_F4', 'sub_gr